
# 04. Cost Budget and SLA Sensitivity
**Goal:** Explore how global cost budgets and per-task latency SLAs constrain the load balancer's decisions.


In [1]:

import sys
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

# Add project root to path
current_dir = Path(os.getcwd())
project_root = current_dir.parent.parent.parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from artemis_final.load_balancer.public_api import (
    ArtemisLoadBalancer,
    ModelCapacityConfig,
    GlobalSLAConfig,
    StatsRegistry,
    RouterOutput,
    SchedulingContext,
    BudgetExhaustedError,
)



## 1. Environment Setup
Models:
- **Cheap**: $0.001
- **Mid**: $0.005
- **Expensive**: $0.02


In [2]:

def setup(budget=10.0, sla_ms=2000.0):
    stats = StatsRegistry()
    task = "vqa"
    
    specs = {
        "cheap_model": (500.0, 0.85, 0.001),
        "mid_model": (300.0, 0.90, 0.005),
        "expensive_model": (100.0, 0.95, 0.02),
    }
    for m, (lat, acc, cost) in specs.items():
        stats.update_latency(task, m, lat)
        stats.update_accuracy(task, m, acc)
        stats.update_cost(task, m, cost)
        
    configs = {m: ModelCapacityConfig(max_rps=50.0, replicas=1) for m in specs.keys()}
    
    # Global Config with Budget
    global_sla = GlobalSLAConfig(total_cost_budget_usd=budget)
    
    lb = ArtemisLoadBalancer(
        model_configs=configs,
        stats_registry=stats,
        latency_sla_ms={"vqa": sla_ms, "default": sla_ms},
        global_sla_config=global_sla,
        scheduling_mode="capacity_aware",
        simulation_only=True
    )
    return lb

print("Setup ready.")


Setup ready.



## 2. Cost Budget Enforcement
We'll run a loop that requests the **Expensive** model (via Router preference) until the budget runs out.
We compare a \$0.10 budget vs a \$1.00 budget.


In [4]:

def run_until_broke(lb, max_reqs=1000):
    reqs_processed = 0
    total_cost = 0.0
    
    # Router prefers expensive model
    router_out = RouterOutput(
        router_probs={"cheap_model": 0.1, "mid_model": 0.1, "expensive_model": 0.8},
        preferred_model="expensive_model",
        max_prob=0.8,
        sample_id='sim_req', task_type='vqa',
    )
    
    try:
        for i in range(max_reqs):
            ctx = SchedulingContext(arrival_ts_ms=time.time() * 1000, sample_id=f"req_{i}", task_type="vqa")
            d = lb.schedule(router_out, ctx)
            total_cost += d.est_cost_usd
            reqs_processed += 1
    except BudgetExhaustedError:
        print(f"💰 Budget Exhausted at request #{reqs_processed}!")
        
    return reqs_processed, total_cost

# Test Small Budget
print("--- Low Budget ($0.10) ---")
lb_low = setup(budget=0.10)
reqs_low, cost_low = run_until_broke(lb_low)
print(f"Processed: {reqs_low}, Cost: ${cost_low:.4f}")

# Test High Budget
print("--- High Budget ($1.00) ---")
lb_high = setup(budget=1.00)
reqs_high, cost_high = run_until_broke(lb_high)
print(f"Processed: {reqs_high}, Cost: ${cost_high:.4f}")

# Plot
plt.figure(figsize=(6,4))
plt.bar(["Low Budget ($0.10)", "High Budget ($1.00)"], [reqs_low, reqs_high], color=['salmon', 'seagreen'])
plt.title("Requests Processed before Exhaustion")
plt.show()


--- Low Budget ($0.10) ---


TypeError: ModelCapacityConfig.__init__() got an unexpected keyword argument 'max_rps'


## 3. SLA Sensitivity
We fix the budget but vary the Per-Task SLA.
- **Loose SLA** (2000ms): Should allow Cheap model (500ms).
- **Strict SLA** (200ms): Should force usage of Expensive (100ms) or Mid (300ms if queue low).


In [ ]:

def run_workload(lb, n=100):
    decisions = []
    # Router prefers Cheap model this time, to test if SLA forces it off
    router_out = RouterOutput(
        router_probs={"cheap_model": 0.8, "mid_model": 0.1, "expensive_model": 0.1},
        preferred_model="cheap_model", # Prefers cheap/slow
        max_prob=0.8,
        sample_id='sim_req', task_type='vqa',
    )
    
    t = 0.0
    for i in range(n):
        t += 0.05
        ctx = SchedulingContext(arrival_ts_ms=t * 1000, sample_id=f"req_{i}", task_type="vqa")
        d = lb.schedule(router_out, ctx)
        decisions.append(d)
        
    return pd.DataFrame([{
        "Chosen": d.chosen_model,
        "Latency": d.total_latency_ms
    } for d in decisions])

# Loose SLA
print("Running Loose SLA (2000ms)...")
lb_loose = setup(budget=100.0, sla_ms=2000.0)
df_loose = run_workload(lb_loose)

# Strict SLA
print("Running Strict SLA (200ms)...")
lb_strict = setup(budget=100.0, sla_ms=200.0) # 200ms target
df_strict = run_workload(lb_strict)

# Compare Usage
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

usage_loose = df_loose['Chosen'].value_counts()
usage_loose.plot(kind='bar', ax=axes[0], color='lightblue')
axes[0].set_title("Model Usage (Loose SLA: 2000ms)")
axes[0].set_xlabel("Router preferred 'cheap_model'")

usage_strict = df_strict['Chosen'].value_counts()
usage_strict.plot(kind='bar', ax=axes[1], color='salmon')
axes[1].set_title("Model Usage (Strict SLA: 200ms)")
axes[1].set_xlabel("Forced to faster models?")

plt.tight_layout()
plt.show()
